# Deploy: llama.cpp OpenAI-compatible server + ngrok

Serve a GGUF (e.g. Gemma 4 12B) and expose `BASE_URL` for [`../eval/10_eval_plausibility.ipynb`](../eval/10_eval_plausibility.ipynb).

**Secrets (CS2309 pattern):** env → Colab `userdata` → widget fallback. Prefer `SHARE_MODE = "ngrok"`.


In [ ]:
# === INPUTS ===
MODEL_ID = "gemma-4-12b"  # logical name for eval notebook MODEL
GGUF_HF_REPO = "bartowski/gemma-2-9b-it-GGUF"  # replace with Gemma-4 GGUF when available
GGUF_FILE = ""  # e.g. "*-Q4_K_M.gguf"; empty = first *Q4_K_M*.gguf found
PORT = 8080
CTX = 4096
N_GPU_LAYERS = 99  # Colab T4: try 20–35 if OOM
SHARE_MODE = "ngrok"  # ngrok | none
API_KEY_DUMMY = "sk-local"


In [ ]:
import os, sys, shutil, subprocess, time
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass
print("IN_COLAB:", IN_COLAB)

def get_secret(name: str, fallback: str = "") -> str:
    val = os.environ.get(name, "")
    if val:
        return val
    if IN_COLAB:
        try:
            from google.colab import userdata
            got = userdata.get(name)
            if got:
                return str(got)
        except Exception as e:
            print("userdata miss:", name, e)
    return fallback

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", "")
print("NGROK token set:", bool(NGROK_AUTHTOKEN))


In [ ]:
# Install deps (Colab-oriented). Local Mac: install llama-server yourself if preferred.
%pip install -q huggingface_hub pyngrok openai

if IN_COLAB:
    # GPU wheel for llama-cpp-python (may take a few minutes)
    %pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/python-wheels/
else:
    %pip install -q llama-cpp-python


In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

cache_dir = Path("/content/gguf_cache") if IN_COLAB else Path.home() / ".cache" / "plausibility_gguf"
cache_dir.mkdir(parents=True, exist_ok=True)

files = list_repo_files(GGUF_HF_REPO)
candidates = [f for f in files if f.endswith(".gguf")]
if GGUF_FILE:
    pick = next((f for f in candidates if GGUF_FILE in f or f == GGUF_FILE), None)
else:
    pick = next((f for f in candidates if "Q4_K_M" in f), None) or (candidates[0] if candidates else None)
if not pick:
    raise FileNotFoundError(f"No GGUF in {GGUF_HF_REPO}")
print("Downloading:", pick)
gguf_path = hf_hub_download(GGUF_HF_REPO, pick, local_dir=str(cache_dir))
print("GGUF:", gguf_path)


In [ ]:
# Start OpenAI-compatible server via llama-cpp-python module
import threading
os.environ["MODEL"] = str(gguf_path)
# Prefer CLI if available
server_proc = None
cmd = [
    sys.executable, "-m", "llama_cpp.server",
    "--model", str(gguf_path),
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--n_ctx", str(CTX),
    "--n_gpu_layers", str(N_GPU_LAYERS),
]
print("Launch:", " ".join(cmd))
server_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# wait for port
import socket
for i in range(60):
    s = socket.socket(); s.settimeout(1)
    try:
        s.connect(("127.0.0.1", PORT)); s.close(); print("Server up on", PORT); break
    except Exception:
        time.sleep(2)
else:
    print("WARN: server may still be loading…")


In [ ]:
# Share via ngrok (preferred) — pattern like CS2309 SwiftEdit notebooks
public_url = None
if SHARE_MODE == "ngrok":
    if not NGROK_AUTHTOKEN:
        raise RuntimeError("Set NGROK_AUTHTOKEN (env / Colab Secret / paste into get_secret fallback)")
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_AUTHTOKEN
    # kill old tunnels
    try:
        ngrok.kill()
    except Exception:
        pass
    tunnel = ngrok.connect(PORT, "http")
    public_url = tunnel.public_url
    print("ngrok:", public_url)
else:
    public_url = f"http://127.0.0.1:{PORT}"
    print("Local only:", public_url)

BASE_URL = public_url.rstrip("/") + "/v1"
print("\n=== Paste into eval notebook ===")
print("MODEL    =", repr(MODEL_ID))
print("TOKEN    =", repr(API_KEY_DUMMY))
print("BASE_URL =", repr(BASE_URL))
print("MODE     = 'ORIG'  # then S / T / ST / ST-E")


In [ ]:
# Smoke: list models / tiny completion
from openai import OpenAI
cli = OpenAI(base_url=BASE_URL, api_key=API_KEY_DUMMY)
print(cli.models.list())
r = cli.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "Reply with a single digit 1-7: how plausible is 'The nurse fetched the patient.'?"}],
    max_tokens=16,
    temperature=0.3,
)
print(r.choices[0].message.content)
